# CHALLENGE 1: CONNECTING TO DATABASE

In [2]:
import redshift_connector

# Create a connection to the Redshift database with the above credentials
conn = redshift_connector.connect(
    host='c23-workgroup.129033205317.eu-west-2.redshift-serverless.amazonaws.com',
    database='dw_air_travel',
    user='yusuf_ahmed',
    password='Dyusuf_71',
    port=5439
)
# Create a cursor to use to interact with the database
cursor = conn.cursor()
# Execute an SQL query
cursor.execute('SELECT Count(Distinct laser_colour) FROM laser_incident')

# Extract the result from the cursor object
output = cursor.fetch_dataframe()

print(output)

# Close the cursor
cursor.close()

   count
0     84


### How many Distinct laser colours are there? 84

In [7]:
# Create a cursor to use to interact with the database
cursor = conn.cursor()
# Execute an SQL query
cursor.execute('SELECT Count(Distinct laser_colour) FROM laser_incident')
# Extract the result from the cursor object
output = cursor.fetch_dataframe()
print(output)
# Close the cursor
cursor.close()

   count
0     84


### How many distinct simplified laser colours? 71

In [3]:
# Create a cursor to use to interact with the database
cursor = conn.cursor()
# Execute an SQL query
cursor.execute('SELECT Count(Distinct (Lower(laser_colour))) FROM laser_incident')
# Extract the result from the cursor object
output = cursor.fetch_dataframe()
print(output)
# Close the cursor
cursor.close()

   count
0     71


## TASK 2: Extracting Data Into Pandas

In [4]:
import pandas as pd
with conn.cursor() as cursor:
    try:
        cursor.execute("""
            SELECT *
            FROM laser_incident
            JOIN airport ON laser_incident.airport_id = airport.airport_id
            JOIN city ON airport.city_id = city.city_id
            JOIN state ON city.state_id = state.state_id
            WHERE state_name = 'Michigan'
        """)

        michigan_laser = cursor.fetch_dataframe()
        print("Number of rows:", len(michigan_laser))

    except Exception as e:
        conn.rollback()
        print(e)

Number of rows: 418


## Task 3
### How many rows are in the dataframe? 418
### What percentage of laser incidents cause significant harm? 0.72%
### What airport has the worst problem with laser incidents? DETROIT METRO WAYNE COUNTY = 176
### How has the frequency of laser incidents changed over time? Decreased slightly over time
### What are the most and least common laser colours used? Green as the Most, Orange as the least

In [5]:
injury = michigan_laser["injury"].value_counts()
print(injury)

injury
f    415
t      3
Name: count, dtype: int64


In [6]:
airport_counts = michigan_laser['airport_name'].value_counts()
print(airport_counts)
michigan_laser.info

airport_name
DETROIT METRO WAYNE COUNTY          176
KALAMAZOO/BATTLE CREEK INTL          37
GERALD R FORD INTL                   35
CAPITAL REGION INTL                  24
BISHOP INTL                          19
MBS INTL                             17
COLEMAN A YOUNG MUNI                 17
OAKLAND COUNTY INTL                  15
CHERRY CAPITAL                       12
MUSKEGON COUNTY                      10
JACKSON COUNTY/REYNOLDS FLD           5
BATTLE CREEK EXEC AT KELLOGG FLD      5
DELTA COUNTY                          4
WILLOW RUN                            4
SELFRIDGE ANGB                        3
PADGHAM FLD                           2
ST CLAIR COUNTY INTL                  2
ALPENA COUNTY RGNL                    2
HILLSDALE MUNI                        2
KIRSCH MUNI                           2
MOUNT PLEASANT MUNI                   2
OTTAWA EXEC                           2
BRANCH COUNTY MEML                    1
CLARE COUNTY                          1
DUPONT/LAPEER              

<bound method DataFrame.info of      laser_incident_id flight_id aircraft  altitude laser_colour injury  \
0                28554    DAL859     A319      8000        Green      f   
1                29885    N350KL     CL35      3000         Blue      f   
2                11781   RPA3527     E170     20000        Green      f   
3                26004     N11SP    HELO       1700        Green      f   
4                17595    DAL853     A321      2000        Green      f   
..                 ...       ...      ...       ...          ...    ...   
413              10699    N383ST     B407      2500         Blue      f   
414              15254   ENY3599     E145      4000        Green      f   
415              27179   SWA1486     B737      4000        Green      f   
416              19509    NKS852     A320      4500        Green      f   
417               3021    N421SC     F2TH      2300         Blue      f   

     airport_id                  at  airport_id                 air

In [7]:
import altair as alt
alt.data_transformers.enable("vegafusion")

michigan_laser = michigan_laser.loc[:, ~michigan_laser.columns.duplicated()]

chart_data = (
    michigan_laser
    .assign(year=michigan_laser['at'].dt.to_period('Y').astype(str))
)

alt.Chart(chart_data).mark_line(point=True).encode(
    x=alt.X('year', title='Year', sort='x'),
    y=alt.Y('count()', title='Total Incidents'),
    tooltip=[
        alt.Tooltip('year', title='Year'),
        alt.Tooltip('count()', title='Total Incidents')
    ]
).properties(
    width=800,
    height=400,
    title='Incidents over time'
)

alt.Chart(...)

In [8]:
alt.Chart(michigan_laser).mark_bar().encode(
    x=alt.X("laser_colour", title="Colour of Laser", sort="-y"),
    y=alt.Y("count()", title="Count"),
    color=alt.Color('laser_colour')
).properties(
    title="Different laser colours",
    width=600,
    height=400)

alt.Chart(...)

# CHALLENGE 2: VISUALISATION

## TASK 1: PIE CHARTS

In [ ]:
with conn.cursor() as cursor:
    try:
        cursor.execute("""
            WITH most_active_airline AS (
                SELECT op_unique_carrier
                FROM flight
                WHERE op_unique_carrier IN (SELECT iata FROM airline)
                GROUP BY op_unique_carrier
                ORDER BY COUNT(*) DESC
                LIMIT 1
            )
            SELECT
                f.tail_num,
                CASE 
                    WHEN COALESCE(CAST(NULLIF(TRIM(f.arr_delay), '') AS FLOAT), 0) > 0
                        THEN 'Delayed'
                    ELSE 'On Time'
                END AS delayed
            FROM flight f
            JOIN most_active_airline m
                ON f.op_unique_carrier = m.op_unique_carrier;
        """)

        flight_delayed = cursor.fetch_dataframe()
        print(flight_delayed)

    except Exception as e:
        conn.rollback()
        print(e)


    

       tail_num  delayed
0        N8526W  On Time
1        N234WN  On Time
2        N420WN  On Time
3        N436WN  On Time
4        N788SA  Delayed
...         ...      ...
127768   N8757L  On Time
127769   N8717M  On Time
127770   N7733B  On Time
127771   N7828A  Delayed
127772   N8822Q  Delayed

[127773 rows x 2 columns]


In [11]:
import altair as alt
delay_counts = flight_delayed['delayed'].value_counts()
delay_percent = (delay_counts / delay_counts.sum()) * 100
delay_percent = delay_percent.reset_index()
delay_percent.columns = ['status', 'percentage']


alt.Chart(delay_percent).mark_arc().encode(
    theta=alt.Theta(field="percentage", type="quantitative"),
    color=alt.Color(field="status", type="nominal"),
    tooltip=[
        alt.Tooltip("status", title="Status"),
        alt.Tooltip("percentage", format=".2f")
    ]
).properties(
    title="Percentage of Flights Delayed (Most Active Airline)"
)

alt.Chart(...)

## Task 2

In [12]:
with conn.cursor() as cursor:
    try:
        cursor.execute("""
            SELECT
                SUM(CAST(NULLIF(TRIM(carrier_delay), '') AS FLOAT)) AS total_carrier_delay,
                SUM(CAST(NULLIF(TRIM(weather_delay), '') AS FLOAT)) AS total_weather_delay,
                SUM(CAST(NULLIF(TRIM(nas_delay), '') AS FLOAT)) AS total_nas_delay,
                SUM(CAST(NULLIF(TRIM(security_delay), '') AS FLOAT)) AS total_security_delay,
                SUM(CAST(NULLIF(TRIM(late_aircraft_delay), '') AS FLOAT)) AS total_late_aircraft_delay
            FROM flight
            WHERE TRIM(cancelled) = '0.0';
        """)
        delay_totals = cursor.fetch_dataframe()
    except Exception as e:
        conn.rollback()
        print(e)

delay_chart_data = pd.DataFrame({
    "delay_type": [
        "Carrier Delay",
        "Weather Delay",
        "NAS Delay",
        "Security Delay",
        "Late Aircraft Delay"
    ],
    "minutes_lost": [
        delay_totals.loc[0, "total_carrier_delay"],
        delay_totals.loc[0, "total_weather_delay"],
        delay_totals.loc[0, "total_nas_delay"],
        delay_totals.loc[0, "total_security_delay"],
        delay_totals.loc[0, "total_late_aircraft_delay"]
    ]
})

# highlight the biggest cause
max_delay = delay_chart_data["minutes_lost"].max()
delay_chart_data["highlight"] = delay_chart_data["minutes_lost"] == max_delay

alt.Chart(delay_chart_data).mark_bar().encode(
    y=alt.Y(
        "delay_type:N",
        sort="-x",
        title=None
    ),
    x=alt.X(
        "minutes_lost:Q",
        title="Total Minutes Lost"
    ),
    color=alt.condition(
        alt.datum.highlight,
        alt.value("#d62728"),
        alt.value("#9ecae1")
    ),
    tooltip=[
        alt.Tooltip("delay_type:N", title="Delay Type"),
        alt.Tooltip("minutes_lost:Q", title="Minutes Lost", format=",")
    ]
).properties(
    width=700,
    height=300,
    title="Total Minutes Lost by Delay Type"
)

alt.Chart(...)